# 01 — Deepnote Confirmatory API Collection

**Do not unlock this notebook until the AsPredicted registration is complete.**

The notebook:
- uses the frozen 14-scenario manifest;
- runs 7 models × 14 scenarios × 2 conditions × 15 successful slots per study;
- retries each failed API call three times (2/5/10 sec);
- if a required call still fails, discards that whole trajectory and starts the slot again from Turn 1 on the next execution;
- checkpoints completed trajectories to JSONL so Deepnote can be stopped/restarted safely.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import os, sys, json, asyncio
from pathlib import Path
import pandas as pd

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
from llm_experiment_core import *

manifest, SCENARIOS, manifest_path = load_manifest()
ROOT = locate_project_root()
RAW = ROOT / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("Working directory:", ROOT)
print("Manifest:", manifest_path)

## Confirmatory lock

After you submit AsPredicted:
1. set `PREREGISTRATION_FROZEN = True`;
2. paste the AsPredicted registration identifier/URL in `AS_PREDICTED_REFERENCE`;
3. set the Deepnote secret/environment variable `CONFIRMATORY_UNLOCK=I_HAVE_PREREGISTERED`.

All three are required.

In [ ]:
PREREGISTRATION_FROZEN = True
AS_PREDICTED_REFERENCE = "https://aspredicted.org/2bd3ri.pdf"
RUN_CONFIRMATORY_API_CALLS = False  # explicit spend-control switch
N_RUNS_PER_CELL = 15
MAX_CONCURRENT_TRAJECTORIES = 4

RUN_METADATA = {
    "aspredicted_reference": AS_PREDICTED_REFERENCE,
    "manifest_sha256": sha256_text(Path(manifest_path).read_text(encoding="utf-8")),
    "package_versions": package_versions(),
    "model_registry": MODEL_REGISTRY,
    "system_prompt": SYSTEM_PROMPT,
    "max_natural_tokens": MAX_NATURAL_TOKENS,
    "max_measurement_tokens": MAX_MEASUREMENT_TOKENS,
    "retry_delays_seconds": RETRY_DELAYS,
}

def assert_confirmatory_ready():
    problems=[]
    if manifest.get("status") != "FROZEN_FOR_PREREGISTRATION":
        problems.append("Scenario manifest status is not FROZEN_FOR_PREREGISTRATION.")
    if not PREREGISTRATION_FROZEN:
        problems.append("PREREGISTRATION_FROZEN is False.")
    if (not AS_PREDICTED_REFERENCE.strip()) or ("PASTE_YOUR" in AS_PREDICTED_REFERENCE):
        problems.append("AS_PREDICTED_REFERENCE has not been filled in.")
    if os.getenv("CONFIRMATORY_UNLOCK") != "I_HAVE_PREREGISTERED":
        problems.append("CONFIRMATORY_UNLOCK secret is not set correctly.")
    if not RUN_CONFIRMATORY_API_CALLS:
        problems.append("RUN_CONFIRMATORY_API_CALLS is False.")
    ks=key_status()
    if not ks["api_key_present"].all():
        problems.append("At least one API key is missing.")
    if problems:
        raise RuntimeError("CONFIRMATORY COLLECTION LOCKED:\n- " + "\n- ".join(problems))

    meta_path = ROOT/"confirmatory_run_metadata.json"
    RUN_METADATA["aspredicted_reference"] = AS_PREDICTED_REFERENCE
    meta_path.write_text(json.dumps(RUN_METADATA, indent=2, default=str), encoding="utf-8")
    print("Confirmatory metadata frozen at:", meta_path)
    return True

print("Preregistration flag is ON. Keep RUN_CONFIRMATORY_API_CALLS=False until you intentionally begin paid confirmatory collection.")

In [ ]:
plan1 = make_plan(1, SCENARIOS, n_runs=N_RUNS_PER_CELL)
plan2 = make_plan(2, SCENARIOS, n_runs=N_RUNS_PER_CELL)

assert len(plan1) == 2940
assert len(plan2) == 2940
display(plan1.head())
print("Study 1 planned successful conversations:", len(plan1))
print("Study 2 planned successful conversations:", len(plan2))

In [ ]:
assert_confirmatory_ready()

## Run Study 1

Safe to rerun: completed experimental slots are skipped. A failed slot is not counted as complete.

In [ ]:
# Confirmatory Study 1
# 1. Fill in AS_PREDICTED_REFERENCE above.
# 2. Set Deepnote secret CONFIRMATORY_UNLOCK=I_HAVE_PREREGISTERED.
# 3. Set RUN_CONFIRMATORY_API_CALLS=True above.
# 4. Then run:

assert_confirmatory_ready()

study1_log = await execute_plan(
    plan1,
    SCENARIOS,
    study=1,
    raw_path=RAW / "study1_confirmatory.jsonl",
    error_path=RAW / "study1_errors.jsonl",
    concurrency=MAX_CONCURRENT_TRAJECTORIES,
)

display(study1_log["status"].value_counts(dropna=False))

In [ ]:
# Progress audit
s1_raw = RAW/"study1_confirmatory.jsonl"
done1 = successful_slots(s1_raw, 1)
print(f"Study 1 successful slots: {len(done1)}/2940")
if len(done1):
    d1 = pd.DataFrame(read_jsonl(s1_raw))
    display(d1.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().describe())

In [ ]:
err1 = RAW / "study1_errors.jsonl"

if err1.exists():
    errors1 = pd.DataFrame(read_jsonl(err1))
    print("Failed trajectory attempts logged:", len(errors1))
    display(errors1["model_key"].value_counts())
else:
    print("No Study 1 failed trajectory attempts logged.")

## Run Study 2

Study 2 contains five natural responses and five separate numerical elicitation forks per successful conversation.

In [ ]:
# Confirmatory Study 2
assert_confirmatory_ready()

study2_log = await execute_plan(
     plan2, SCENARIOS, study=2,
     raw_path=RAW/"study2_confirmatory.jsonl",
     error_path=RAW/"study2_errors.jsonl",
     concurrency=MAX_CONCURRENT_TRAJECTORIES,
)

display(study2_log["status"].value_counts(dropna=False))

In [ ]:
# Progress audit
s2_raw = RAW/"study2_confirmatory.jsonl"
done2 = successful_slots(s2_raw, 2)
print(f"Study 2 successful slots: {len(done2)}/2940")
if len(done2):
    d2 = pd.DataFrame(read_jsonl(s2_raw))
    display(d2.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().describe())

In [ ]:
# Study 2 failed-attempt audit
err2 = RAW / "study2_errors.jsonl"

if err2.exists():
    errors2 = pd.DataFrame(read_jsonl(err2))
    print("Failed trajectory attempts logged:", len(errors2))
    display(errors2["model_key"].value_counts())
else:
    print("No Study 2 failed trajectory attempts logged.")

## Provider metadata audit

Adapted from the original pilot: confirm the provider-reported model IDs, response IDs and token metadata before analysis.

In [ ]:
def provider_metadata_audit():
    frames=[]
    for f in [RAW/"study1_confirmatory.jsonl", RAW/"study2_confirmatory.jsonl"]:
        if f.exists():
            frames.append(pd.DataFrame(read_jsonl(f)))
    if not frames:
        print("No confirmatory data yet.")
        return
    d=pd.concat(frames,ignore_index=True)
    summary=(d.groupby(["model_key","requested_model"])
              .agg(
                  natural_reported_models=("natural_reported_model", lambda x: sorted(set(str(v) for v in x.dropna()))),
                  measurement_reported_models=("measurement_reported_model", lambda x: sorted(set(str(v) for v in x.dropna()))),
                  mean_natural_input_tokens=("natural_input_tokens","mean"),
                  mean_natural_output_tokens=("natural_output_tokens","mean"),
                  mean_measurement_input_tokens=("measurement_input_tokens","mean"),
                  mean_measurement_output_tokens=("measurement_output_tokens","mean"),
                  natural_retry_rate=("natural_api_attempts", lambda x: (x>1).mean()),
                  measurement_retry_rate=("measurement_api_attempts", lambda x: (x>1).mean()),
              ).reset_index())
    display(summary)

provider_metadata_audit()

## Final collection integrity check

Run after both studies reach 2,940 successful slots.

In [ ]:
def final_collection_audit():
    d1 = pd.DataFrame(read_jsonl(RAW/"study1_confirmatory.jsonl"))
    d2 = pd.DataFrame(read_jsonl(RAW/"study2_confirmatory.jsonl"))
    assert len(successful_slots(RAW/"study1_confirmatory.jsonl",1)) == 2940
    assert len(successful_slots(RAW/"study2_confirmatory.jsonl",2)) == 2940
    assert len(d1) == 2940
    assert len(d2) == 2940*5
    assert d1.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().eq(15).all()
    assert d2.groupby(["model_key","scenario_id","condition"])["slot_id"].nunique().eq(15).all()
    assert d2.groupby("slot_id")["turn"].nunique().eq(5).all()
    print("PASS: exact confirmatory sample achieved.")
    print("Study 1 rows:",len(d1),"Study 2 turn rows:",len(d2))

final_collection_audit()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=50cf22b4-6086-46b7-bde8-9b24fcd40297' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>